# Lab 0B: Compare Three Models on PII Extraction

Use this notebook after you finish `../lab0_1_environment_setup/03_environment_check.ipynb`.

In this warm-up notebook, you will use the configured model plus **two other models** from the same Ollama endpoint and compare how well they extract PII from a short synthetic case note.

This task is a good fit for Lab 0 because it is:
- easy to run
- relevant to digital forensics
- structured enough to score consistently
- based on synthetic data rather than real personal data

In [ ]:
import json
from pathlib import Path
from time import perf_counter

import requests
from dotenv import dotenv_values
from openai import OpenAI

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / '.env.example').exists() and (candidate / 'src').exists():
            return candidate
    raise FileNotFoundError('Could not find the repo root.')

repo_root = find_repo_root(Path.cwd().resolve())
config = dotenv_values(repo_root / '.env')
default_model = config.get('MODEL')
ollama_base_url = config.get('OLLAMA_BASE_URL')

if not default_model or not ollama_base_url:
    raise ValueError('MODEL or OLLAMA_BASE_URL is missing from .env')

client = OpenAI(base_url=ollama_base_url, api_key='ollama')

print('Repo root:', repo_root)
print('Default model from .env:', default_model)
print('Ollama endpoint:', ollama_base_url)

## Step 1: Discover Available Models

Run the next cell to list the models available from the configured Ollama endpoint.

In [ ]:
api_root = ollama_base_url.rstrip('/')
if api_root.endswith('/v1'):
    tags_url = api_root[:-3] + '/api/tags'
else:
    tags_url = api_root + '/api/tags'

response = requests.get(tags_url, timeout=10)
response.raise_for_status()
available_models = [item.get('name') for item in response.json().get('models', []) if item.get('name')]

print('Available models:')
for index, model_name in enumerate(available_models, start=1):
    print(f'{index}. {model_name}')

if len(available_models) < 3:
    raise ValueError('This homework needs at least 3 available models.')

## Step 2: Choose Three Models

Use the default model from `.env`, then choose **two different models** from the list above.

The next cell gives you a starting point. Edit the list if you want different comparison models.

In [ ]:
other_models = [name for name in available_models if name != default_model]
models_to_compare = [default_model] + other_models[:2]

print('Models selected for comparison:')
for model_name in models_to_compare:
    print('-', model_name)

if len(set(models_to_compare)) != 3:
    raise ValueError('Choose 3 different models before continuing.')

## Step 3: Synthetic Case Note

Use the same synthetic case note for all three models.

In [ ]:
case_note = """
Case note:
Investigator Maya Chen documented an interview with Jordan Lee at 2458 West Pine Street, Springfield, IL 62704.
Jordan Lee said a suspicious text came from 415-555-0187 and referenced alex.romero88@example.com.
A second contact for follow-up was Priya Nair at 202-555-0142 and priyanair@sample.org.
The seized phone record listed IMEI 356938035643809 and serial number SN-A19XZ-4421.
""".strip()

print(case_note)

## Step 4: Shared Extraction Prompt

All three models should answer the **same prompt** so you can compare them fairly.

In [ ]:
comparison_prompt = f"""
Extract PII from the synthetic case note below.

Return valid JSON only.
Use exactly these keys:
- names
- phone_numbers
- email_addresses
- physical_addresses
- device_ids

Rules:
- Keep each item exactly as it appears in the note.
- Use arrays for all five keys.
- Do not include explanations.
- Do not add keys beyond the five listed above.

Case note:
{case_note}
""".strip()

print(comparison_prompt)

In [ ]:
def clean_json_text(text: str) -> str:
    cleaned = text.strip()
    if cleaned.startswith('```'):
        cleaned = cleaned.strip('`')
        cleaned = cleaned.replace('json\n', '', 1).strip()
    start = cleaned.find('{')
    end = cleaned.rfind('}')
    if start != -1 and end != -1 and end > start:
        cleaned = cleaned[start:end + 1]
    return cleaned

def ask_model(model_name: str, prompt: str) -> dict:
    start = perf_counter()
    response = client.chat.completions.create(
        model=model_name,
        messages=[{'role': 'user', 'content': prompt}]
    )
    elapsed = perf_counter() - start
    raw_text = response.choices[0].message.content
    cleaned_text = clean_json_text(raw_text)

    try:
        parsed = json.loads(cleaned_text)
        parse_error = None
    except Exception as exc:
        parsed = None
        parse_error = str(exc)

    return {
        'model': model_name,
        'seconds': round(elapsed, 2),
        'raw_text': raw_text,
        'parsed': parsed,
        'parse_error': parse_error,
    }

results = []
for model_name in models_to_compare:
    print(f'Running {model_name}...')
    results.append(ask_model(model_name, comparison_prompt))

print('Comparison complete.')

In [ ]:
for result in results:
    print('=' * 80)
    print('Model:', result['model'])
    print('Time (seconds):', result['seconds'])
    print('Parse error:', result['parse_error'])
    print('-' * 80)
    print(result['raw_text'])
    print()

## Step 5: Score the Results

This rubric is easier to measure than broad questions like "Which answer was best?"

Scoring rules:
- 1 point if the model returns valid JSON
- 1 point if `names` matches the expected set
- 1 point if `phone_numbers` matches the expected set
- 1 point if `email_addresses` matches the expected set
- 1 point if `physical_addresses` matches the expected set
- 1 point if `device_ids` matches the expected set

In [ ]:
expected = {
    'names': {'Maya Chen', 'Jordan Lee', 'Priya Nair'},
    'phone_numbers': {'415-555-0187', '202-555-0142'},
    'email_addresses': {'alex.romero88@example.com', 'priyanair@sample.org'},
    'physical_addresses': {'2458 West Pine Street, Springfield, IL 62704'},
    'device_ids': {'IMEI 356938035643809', 'SN-A19XZ-4421'},
}

def normalize_items(value):
    if isinstance(value, list):
        return {str(item).strip() for item in value}
    return set()

def score_result(result: dict) -> dict:
    parsed = result['parsed'] if isinstance(result['parsed'], dict) else {}
    checks = {
        'valid_json': result['parsed'] is not None,
        'names_match': normalize_items(parsed.get('names')) == expected['names'],
        'phone_numbers_match': normalize_items(parsed.get('phone_numbers')) == expected['phone_numbers'],
        'email_addresses_match': normalize_items(parsed.get('email_addresses')) == expected['email_addresses'],
        'physical_addresses_match': normalize_items(parsed.get('physical_addresses')) == expected['physical_addresses'],
        'device_ids_match': normalize_items(parsed.get('device_ids')) == expected['device_ids'],
    }
    checks['total_score'] = sum(int(value) for value in checks.values())
    return {**result, **checks}

scored_results = [score_result(result) for result in results]

for result in scored_results:
    print('=' * 80)
    print('Model:', result['model'])
    print('Time (seconds):', result['seconds'])
    print('Total score:', result['total_score'], '/ 6')
    print('valid_json:', result['valid_json'])
    print('names_match:', result['names_match'])
    print('phone_numbers_match:', result['phone_numbers_match'])
    print('email_addresses_match:', result['email_addresses_match'])
    print('physical_addresses_match:', result['physical_addresses_match'])
    print('device_ids_match:', result['device_ids_match'])
    print()

## Reflection Questions

Replace this text with short answers to the questions below.

Use the scores above to support your answers.

1. Which two additional models did you choose?
2. Which model earned the highest total score?
3. Which model was the fastest?
4. Which exact PII category did the lowest-scoring model miss?
5. Did any model fail to return valid JSON?
6. If you were doing a simple PII extraction task, which model would you choose and why?

## Submission

Save the notebook with:
- the discovered model list
- the three selected models
- the raw outputs from all three models
- the score summary
- your short reflection